# Zero-shot text-to-video benchmark on ActivityNet Captions

Frozen image and video language models, scored zero-shot on ActivityNet Captions. Every caption is a query; its only positive is the gallery video it describes.

| Dataset | Gallery | Queries | What the captions describe |
|---|---|---|---|
| **ActivityNet Captions** (`paragraph`) | `val1` videos (random sample of `MAX_VIDEOS`, or all 4,917) | 1 per video: the sentences of the video joined into one paragraph | the whole video (long-form, ~180 s on average) |
| **ActivityNet Captions** (`sentence`) | same gallery | one per annotated event sentence (about 3.6 per video) | one event of the video |

Source: [`friedrichor/ActivityNet_Captions`](https://huggingface.co/datasets/friedrichor/ActivityNet_Captions) (official splits: train 10,009 / val1 4,917 / val2 4,885 videos). `val1` is the usual paragraph-to-video test split. Set `ANET_SPLIT = "val2"` to use the other one.

**How to run**

1. Kaggle GPU T4 x2, Internet on. Add a Kaggle secret named `hugging_face` and attach it to the notebook (used for the dataset download).
2. Pick `ENV_GROUP` in the configuration cell. The groups need different `transformers` versions, so each group is its own Kaggle session:
   - `base`: SigLIP 2, Perception Encoder, IRRA, X-CLIP
   - `extended_modern`: OpenAI CLIP ViT-L/14, Jina CLIP v2, Qwen3-VL-Embedding-2B
   - `extended_gme`: GME-Qwen2-VL-2B
3. Run All. On Kaggle, two workers run models concurrently, one pinned to each T4. Every result row is written as JSON the moment it finishes, so a restarted session skips what is done.
4. To combine groups, save each session's `/kaggle/working/zero_shot_results` as a Kaggle dataset and attach it; the tables section merges every results folder it finds.

**Time and disk.** ActivityNet videos are long, so the sliding-window protocols are expensive. `MAX_VIDEOS` (default 1000, seeded) keeps a session inside the Kaggle time limit; set it to `None` for the full split. The videos live inside a 42 GB split archive on the Hub, which is streamed once and only the selected videos are kept on disk.

## 1. Configuration

In [1]:
from pathlib import Path

ENV_GROUP = "base"  # "base", "extended_modern", or "extended_gme" (one Kaggle session each)

MODEL_GROUPS = {
    "base": ["siglip2", "pe_core_l14", "irra", "xclip"],
    "extended_modern": ["openai_clip_vit_l14", "jina_clip_v2", "qwen3_vl_embed_2b"],
    "extended_gme": ["gme_qwen2_vl_2b"],
}
MODELS = MODEL_GROUPS[ENV_GROUP]

DATASETS = {
    "activitynet": {"label": "ActivityNet Captions", "subsets": ["paragraph", "sentence"], "protocols": "video"},
}
SPLIT = "test"

# ActivityNet Captions: "val1" is the standard paragraph-to-video test split, "val2" is the other one.
ANET_SPLIT = "val1"
MAX_VIDEOS = 1000  # random gallery sample (seeded); None keeps every video of the split
ANET_SEED = 0
DEVICE = "cuda"  # two visible GPUs run one model worker each
BATCH = 4
TEXT_BATCH = 16
POOLS = ("mean", "mean_s8", "max", "query_max")

# 1B-s2 is built for 4-frame clips; every other model uses 8.
NATIVE_FRAMES = {"internvideo2_s2_1b": 4}
MODEL_BATCH = {
    "siglip2": 1,
    "pe_core_l14": 1,
    "irra": 2,
    "internvideo2_s2_1b": 1,
    "internvideo2": 2,
    "languagebind": 2,
    "openai_clip_vit_l14": 4,
    "jina_clip_v2": 1,
    "gme_qwen2_vl_2b": 1,
    "qwen3_vl_embed_2b": 1,
}
MODEL_TEXT_BATCH = {"jina_clip_v2": 8, "gme_qwen2_vl_2b": 8, "qwen3_vl_embed_2b": 1}

SINGLE_PROTOCOLS = [
    {"name": "uniform8", "num_frames": 8, "frame_sample": "uniform", "family": "clip8"},
    {"name": "uniform4", "num_frames": 4, "frame_sample": "uniform", "family": "s2"},
    {"name": "middle4", "num_frames": 4, "frame_sample": "middle", "family": "s2", "video_only": True},
]
WINDOW_PROTOCOLS = [
    {"name": "vt_1fps_n12", "sample_fps": 1.0, "max_frames": 12, "stride": 4},
    {"name": "vt_2fps_n32", "sample_fps": 2.0, "max_frames": 32, "stride": 4},
    {"name": "reid_8fps_n64", "sample_fps": 8.0, "max_frames": 64, "stride": 4},
]

PRETTY = {
    "irra": "IRRA",
    "pe_core_l14": "Perception Encoder L/14",
    "siglip2": "SigLIP 2 So400m",
    "internvideo2": "InternVideo2 CLIP-S",
    "internvideo2_s2_1b": "InternVideo2-1B-s2",
    "languagebind": "LanguageBind",
    "xclip": "X-CLIP",
    "openai_clip_vit_l14": "OpenAI CLIP L/14",
    "jina_clip_v2": "Jina CLIP v2",
    "gme_qwen2_vl_2b": "GME-Qwen2-VL-2B",
    "qwen3_vl_embed_2b": "Qwen3-VL-Embedding-2B",
}

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("results")
# Videos and frame JPEGs are large and disposable, so they go to /kaggle/temp when it exists;
# only the small result JSONs stay in /kaggle/working.
DATA_ROOT = WORK
if Path("/kaggle/working").is_dir():
    try:
        Path("/kaggle/temp").mkdir(parents=True, exist_ok=True)
        DATA_ROOT = Path("/kaggle/temp")
    except OSError:
        DATA_ROOT = WORK
RESULTS_ROOT = WORK / "zero_shot_results"
RESULTS_DIR = RESULTS_ROOT / ENV_GROUP
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Group", ENV_GROUP, "models", MODELS, flush=True)
print("Dataset ActivityNet Captions", "results", RESULTS_DIR, "data", DATA_ROOT, flush=True)

Group base models ['siglip2', 'pe_core_l14', 'irra', 'xclip']
Dataset ActivityNet Captions results /kaggle/working/zero_shot_results/base data /kaggle/temp


## 2. Setup

Clones the repo (and OpenGVLab/InternVideo when 1B-s2 is in the group), installs the group's dependencies, checks the package version, and loads the Hugging Face token. If `transformers` was already imported with a different version, restart the session and run again.

In [3]:
import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

SHAWAF_URL = "https://github.com/BASSAT-BASSAT/Benchmarking-Video-Image-language-models-for-Tracklet-retrieval-.git"
INTERNVIDEO_URL = "https://github.com/OpenGVLab/InternVideo.git"
EXPECTED = "0.1.18"
TORCH_INDEX = "https://download.pytorch.org/whl/cu126"


def run(cmd, cwd=None):
    print("+", " ".join(map(str, cmd)), flush=True)
    subprocess.check_call([str(part) for part in cmd], cwd=str(cwd) if cwd else None)


def installed(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None


def sync_repo(url, path):
    if path.exists():
        run(["git", "fetch", "origin"], cwd=path)
        run(["git", "reset", "--hard", "origin/main"], cwd=path)
    else:
        run(["git", "clone", url, str(path)])


NEEDS_INTERNVIDEO = "internvideo2_s2_1b" in MODELS
ON_KAGGLE = Path("/kaggle/working").is_dir()
transformers_before = installed("transformers")
transformers_loaded = "transformers" in sys.modules
torch_before = installed("torch")
torch_loaded = "torch" in sys.modules

if ON_KAGGLE:
    shawaf_dir = Path("/kaggle/working/shawaf-vlm")
    intern_dir = Path("/kaggle/working/InternVideo")
    sync_repo(SHAWAF_URL, shawaf_dir)
    if NEEDS_INTERNVIDEO:
        sync_repo(INTERNVIDEO_URL, intern_dir)
    if ENV_GROUP == "base":
        run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{shawaf_dir}[all]"])
        run(
            [
                sys.executable, "-m", "pip", "install", "-q",
                "einops", "timm", "av", "imageio", "librosa", "soundfile",
                "pandas", "pyyaml", "scipy", "wandb", "gdown",
            ]
        )
        try:
            run([sys.executable, "-m", "pip", "install", "-q", "decord"])
        except subprocess.CalledProcessError:
            print("decord wheel failed; the 1B-s2 path stubs it and uses PyAV", flush=True)
    else:
        if not (installed("torch") or "").startswith("2.8.0"):
            run([sys.executable, "-m", "pip", "install", "-q", "--index-url", TORCH_INDEX, "torch==2.8.0", "torchvision==0.23.0"])
        run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{shawaf_dir}[{ENV_GROUP}]", "gdown>=5.2", "pandas"])
else:
    here = Path.cwd().resolve()
    shawaf_dir = here
    for candidate in [here, *here.parents]:
        if (candidate / "shawaf_vlm").is_dir() and (candidate / "pyproject.toml").is_file():
            shawaf_dir = candidate
            break
    intern_dir = shawaf_dir / "InternVideo"
    if NEEDS_INTERNVIDEO and not intern_dir.is_dir():
        run(["git", "clone", INTERNVIDEO_URL, str(intern_dir)], cwd=shawaf_dir)

transformers_after = installed("transformers")
if transformers_loaded and transformers_after != transformers_before:
    raise RuntimeError(
        f"transformers changed from {transformers_before} to {transformers_after} after it was "
        "imported. Restart the session, then Run All again."
    )

torch_after = installed("torch")
print("torch", torch_after, flush=True)
if torch_loaded and torch_after != torch_before:
    raise RuntimeError(
        f"torch changed from {torch_before} to {torch_after} after it was imported. "
        "Restart the session, then Run All again."
    )

if NEEDS_INTERNVIDEO:
    os.environ["INTERNVIDEO_ROOT"] = str(intern_dir)
if str(shawaf_dir) not in sys.path:
    sys.path.insert(0, str(shawaf_dir))
for name in list(sys.modules):
    if name == "shawaf_vlm" or name.startswith("shawaf_vlm."):
        del sys.modules[name]

import shawaf_vlm

print("shawaf_vlm", shawaf_vlm.__version__, shawaf_vlm.__file__, flush=True)
print("transformers", transformers_after, flush=True)
if shawaf_vlm.__version__ != EXPECTED:
    raise RuntimeError(
        f"Expected shawaf_vlm {EXPECTED}, found {shawaf_vlm.__version__}. "
        "Restart the session and run this cell again after origin/main updates."
    )

token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if not token and ON_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient

        token = UserSecretsClient().get_secret("hugging_face")
    except Exception as exc:
        print("No hugging_face secret:", type(exc).__name__, flush=True)
if token:
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    from huggingface_hub import login

    login(token=token, add_to_git_credential=False)
    print("Hugging Face token loaded", flush=True)
elif NEEDS_INTERNVIDEO:
    raise RuntimeError(
        "Hugging Face token missing. InternVideo2-1B-s2 is gated: add a Kaggle secret "
        "named hugging_face and attach it to this notebook."
    )
else:
    print("No Hugging Face token; public datasets and models only", flush=True)

+ git fetch origin
+ git reset --hard origin/main
HEAD is now at d553da9 evaluated initial models on our 3 datasets
+ /usr/bin/python3 -m pip install -q -e /kaggle/working/shawaf-vlm[all]
+ /usr/bin/python3 -m pip install -q einops timm av imageio librosa soundfile pandas pyyaml scipy wandb gdown
+ /usr/bin/python3 -m pip install -q decord
torch 2.11.0+cu128
shawaf_vlm 0.1.18 /kaggle/working/shawaf-vlm/shawaf_vlm/__init__.py
transformers 5.16.1


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face token loaded


## 3. Datasets

ActivityNet Captions ships its videos inside `ActivityNet_Videos.tar.part-000 ... 007` (about 42 GB). The next cell reads the split JSON, draws the gallery (`MAX_VIDEOS` seeded random `ANET_SPLIT` videos), streams the archive parts one after another (the next part downloads while the current one is read, and a finished part is deleted at once), keeps only the selected videos, and writes `metadata/paragraph-test.jsonl` and `metadata/sentence-test.jsonl`. A rerun finds the videos already extracted and downloads nothing.

Both subsets share one gallery, so gallery embeddings are reused between them (rows marked `gallery_cached`).

In [4]:
import io
import json
import os
import random
import shutil
import tarfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
from huggingface_hub import hf_hub_download
from IPython.display import Markdown, display
from tqdm.auto import tqdm

from shawaf_vlm.data.hub_video import load_hub_video_split

ANET_REPO = "friedrichor/ActivityNet_Captions"
ANET_PARTS = [f"ActivityNet_Videos.tar.part-{i:03d}" for i in range(8)]
ANET_ROOT = DATA_ROOT / "ActivityNet_Captions"
ANET_VIDEOS = ANET_ROOT / "videos"
ANET_META = ANET_ROOT / "metadata"
ANET_PARTS_DIR = ANET_ROOT / "_parts"


class PartStream(io.RawIOBase):
    """Read the split tar as one byte stream, downloading one part ahead and deleting finished parts."""

    def __init__(self, names, dest, token=None):
        self.names = list(names)
        self.dest = Path(dest)
        self.token = token
        self.dest.mkdir(parents=True, exist_ok=True)
        self.pool = ThreadPoolExecutor(max_workers=1)
        self.index = -1
        self.handle = None
        self.current = None
        self.pending = None
        self._schedule(0)

    def _fetch(self, name):
        return Path(
            hf_hub_download(
                repo_id=ANET_REPO,
                filename=name,
                repo_type="dataset",
                token=self.token,
                local_dir=str(self.dest),
            )
        )

    def _schedule(self, index):
        self.pending = self.pool.submit(self._fetch, self.names[index]) if index < len(self.names) else None

    def _open_next(self):
        self.index += 1
        future = self.pending
        if future is None:
            return False
        path = future.result()
        self._schedule(self.index + 1)
        self.current = path
        self.handle = path.open("rb")
        print(f"reading {path.name} ({self.index + 1}/{len(self.names)})", flush=True)
        return True

    def _finish_part(self):
        if self.handle is not None:
            self.handle.close()
            self.handle = None
        if self.current is not None:
            self.current.unlink(missing_ok=True)
            self.current = None

    def readable(self):
        return True

    def readinto(self, buffer):
        while True:
            if self.handle is None and not self._open_next():
                return 0
            count = self.handle.readinto(buffer)
            if count:
                return count
            self._finish_part()

    def close(self):
        if not self.closed:
            self._finish_part()
            if self.pending is not None:
                self.pending.cancel()
            self.pool.shutdown(wait=True)
            shutil.rmtree(self.dest, ignore_errors=True)
        super().close()


def anet_select_rows():
    """Read the split JSON and draw the gallery (seeded, order kept)."""
    path = hf_hub_download(
        repo_id=ANET_REPO,
        filename=f"activitynet_captions_{ANET_SPLIT}.json",
        repo_type="dataset",
        token=os.environ.get("HF_TOKEN"),
    )
    rows = json.loads(Path(path).read_text(encoding="utf-8"))
    seen, unique = set(), []
    for row in rows:
        if row["video_id"] not in seen:
            seen.add(row["video_id"])
            unique.append(row)
    if MAX_VIDEOS is not None and len(unique) > MAX_VIDEOS:
        picked = sorted(random.Random(ANET_SEED).sample(range(len(unique)), MAX_VIDEOS))
        unique = [unique[i] for i in picked]
    return unique


def anet_local_videos():
    if not ANET_VIDEOS.is_dir():
        return {}
    return {p.stem: p for p in ANET_VIDEOS.iterdir() if p.is_file() and p.suffix != ".tmp"}


def anet_extract(video_ids):
    """Stream the split archive once and keep only ``video_ids`` (matched on file stem)."""
    remaining = set(video_ids)
    ANET_VIDEOS.mkdir(parents=True, exist_ok=True)
    stream = PartStream(ANET_PARTS, ANET_PARTS_DIR, token=os.environ.get("HF_TOKEN"))
    bar = tqdm(total=len(remaining), desc="Extract videos", unit="video")
    try:
        reader = io.BufferedReader(stream, buffer_size=8 * 1024 * 1024)
        with tarfile.open(fileobj=reader, mode="r|") as archive:
            for member in archive:
                if not member.isfile():
                    continue
                name = Path(member.name).name
                stem = Path(name).stem
                if stem not in remaining:
                    continue
                target = ANET_VIDEOS / name
                tmp = target.with_name(target.name + ".tmp")
                with archive.extractfile(member) as source, tmp.open("wb") as out:
                    shutil.copyfileobj(source, out, 4 * 1024 * 1024)
                os.replace(tmp, target)
                remaining.discard(stem)
                bar.update(1)
                if not remaining:
                    break
    finally:
        bar.close()
        stream.close()
    if remaining:
        print(f"{len(remaining)} selected videos were not found in the archive", flush=True)


def anet_write_metadata(rows, have):
    ANET_META.mkdir(parents=True, exist_ok=True)
    kept = [row for row in rows if row["video_id"] in have]
    for subset in DATASETS["activitynet"]["subsets"]:
        with (ANET_META / f"{subset}-{SPLIT}.jsonl").open("w", encoding="utf-8") as handle:
            for row in kept:
                if subset == "paragraph":
                    captions = [row["caption"].strip()]
                elif subset == "sentence":
                    captions = [s.strip() for s in row["sentences"] if s and s.strip()]
                else:
                    raise ValueError(f"Unknown ActivityNet subset {subset!r}")
                record = {
                    "video": f"videos/{have[row['video_id']].name}",
                    "video_id": row["video_id"],
                    "captions": captions,
                    "subset": ANET_SPLIT,
                }
                handle.write(json.dumps(record, ensure_ascii=False) + "\n")
    return len(kept)


def prepare_activitynet():
    rows = anet_select_rows()
    have = anet_local_videos()
    missing = [row["video_id"] for row in rows if row["video_id"] not in have]
    print(f"ActivityNet {ANET_SPLIT}: {len(rows)} selected videos, {len(rows) - len(missing)} already on disk", flush=True)
    if missing:
        anet_extract(missing)
        have = anet_local_videos()
    kept = anet_write_metadata(rows, have)
    print(f"Wrote metadata for {kept} videos under {ANET_META}", flush=True)
    return ANET_ROOT


# Hugging Face dataset import for Kaggle: download the selected split JSON and
# stream its video archive directly from the dataset repository.
DATA_ROOTS = {"activitynet": prepare_activitynet()}


def load_split(subset):
    return load_hub_video_split(
        DATA_ROOTS["activitynet"],
        config=subset,
        split=SPLIT,
        source=f"activitynet:{ANET_SPLIT}:{subset}",
        missing=lambda detail: FileNotFoundError(detail),
        name="ActivityNet Captions",
    )


summary = []
for subset in DATASETS["activitynet"]["subsets"]:
    splits = load_split(subset)
    per_id = pd.Series([g.person_id for g in splits.gallery]).value_counts()
    positives = pd.Series([per_id.get(q.person_id, 0) for q in splits.query])
    summary.append(
        {
            "dataset": "ActivityNet Captions",
            "subset": subset,
            "queries": len(splits.query),
            "gallery": len(splits.gallery),
            "identities": int(per_id.size),
            "positives / query": round(float(positives.mean()), 2),
            "root": str(DATA_ROOTS["activitynet"]),
        }
    )
DATASET_SUMMARY = pd.DataFrame(summary)
display(DATASET_SUMMARY)


activitynet_captions_val1.json:   0%|          | 0.00/5.23M [00:00<?, ?B/s]

ActivityNet val1: 1000 selected videos, 0 already on disk


Extract videos:   0%|          | 0/1000 [00:00<?, ?video/s]

ActivityNet_Videos.tar.part-000: reconstructing file:   0%|          |  0.00B / 5.37GB            

ActivityNet_Videos.tar.part-000: downloading bytes:           |  0.00B            

...

reading ActivityNet_Videos.tar.part-006 (7/8)


ActivityNet_Videos.tar.part-007: reconstructing file:   0%|          |  0.00B / 4.40GB            

ActivityNet_Videos.tar.part-007: downloading bytes:           |  0.00B            

reading ActivityNet_Videos.tar.part-007 (8/8)
Wrote metadata for 1000 videos under /kaggle/temp/ActivityNet_Captions/metadata


,dataset,subset,queries,gallery,identities,positives / query,root
0,ActivityNet Captions,paragraph,1000,1000,1000,1.0,/kaggle/temp/ActivityNet_Captions
1,ActivityNet Captions,sentence,3617,1000,1000,1.0,/kaggle/temp/ActivityNet_Captions


## 4. Run

Each GPU worker takes the next model, then runs dataset, protocol, and subset in order. Each finished row is saved to `zero_shot_results/<group>/<dataset>/<model>_<subset>_<protocol>_<pool>.json` and skipped on rerun.

The ActivityNet `paragraph` and `sentence` subsets share one gallery, so gallery embeddings are reused between them within a protocol (those rows are marked `gallery_cached`; speed is read from the uncached row).

In [5]:
import torch
print(torch.__file__)
print(hasattr(torch, "device"), torch.__version__)

/usr/local/lib/python3.13/dist-packages/torch/__init__.py
True 2.11.0+cu128


In [6]:
import gc
import json
import traceback
from concurrent.futures import ThreadPoolExecutor
from queue import Empty, Queue
from datetime import datetime, timezone

from shawaf_vlm.eval_loop import evaluate_text_to_tracklet, evaluate_text_to_tracklet_windows
from shawaf_vlm.metrics import format_metrics
from shawaf_vlm.models import build_encoder
from shawaf_vlm.models.runtime import ensure_cuda_healthy

METRIC_KEYS = [
    "Rank-1", "Rank-5", "Rank-10", "Rank-20", "Rank-50",
    "mAP", "MdR", "MnR", "nDCG@10", "mINP",
    "num_valid_queries", "num_gallery", "num_clips",
    "decode_s", "video_s", "text_s", "score_s", "total_s",
    "video_ms_per_item", "peak_gpu_gb", "reserved_gpu_gb",
]


class StopModel(Exception):
    """CUDA OOM: stop this model, keep benchmark settings unchanged."""


class GalleryCache:
    """Reuse the last gallery embedding when the exact same frame lists come back."""

    def __init__(self, encoder):
        self.encoder = encoder
        self._key = None
        self._features = None
        self.last_hit = False

    def encode_videos(self, videos, batch_size=4):
        key = tuple(tuple(str(path) for path in clip) for clip in videos)
        if key == self._key:
            self.last_hit = True
            return self._features
        self.last_hit = False
        self._features = self.encoder.encode_videos(videos, batch_size=batch_size)
        self._key = key
        return self._features

    def encode_texts(self, texts, batch_size=32):
        return self.encoder.encode_texts(texts, batch_size=batch_size)

    def __getattr__(self, name):
        return getattr(self.encoder, name)


def family(model):
    return "s2" if model in NATIVE_FRAMES else "clip8"


def protocols_for(model):
    kind = "video"
    specs = []
    for spec in SINGLE_PROTOCOLS:
        if spec["family"] != family(model):
            continue
        if kind == "image" and spec.get("video_only"):
            continue
        specs.append({**spec, "type": "single"})
    if kind == "video":
        specs.extend({**spec, "type": "window"} for spec in WINDOW_PROTOCOLS)
    return specs


def sampling_text(model, spec):
    if spec["type"] == "single":
        return f"{spec['frame_sample']} {spec['num_frames']} frames"
    return (
        f"sliding windows of {NATIVE_FRAMES.get(model, 8)}, stride {spec['stride']}, "
        f"{spec['sample_fps']:g} fps, cap {spec['max_frames']} frames"
    )


def result_path(dataset, model, subset, protocol, pool):
    return RESULTS_DIR / dataset / f"{model}_{subset}_{protocol}_{pool}.json"


def expected_paths(dataset, model, subset, spec):
    pools = ["none"] if spec["type"] == "single" else list(POOLS)
    return [result_path(dataset, model, subset, spec["name"], pool) for pool in pools]


def is_complete(path):
    try:
        return json.loads(path.read_text(encoding="utf-8")).get("status") == "complete"
    except Exception:
        return False


def save_row(dataset, model, subset, spec, pool, metrics=None, error=None, cached=False):
    row = {
        "group": ENV_GROUP,
        "dataset": dataset,
        "subset": subset,
        "model": model,
        "protocol": spec["name"],
        "protocol_type": spec["type"],
        "pool": pool,
        "sampling": sampling_text(model, spec),
        "gallery_cached": bool(cached),
        "status": "failed" if error else "complete",
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    if metrics is not None:
        row.update({key: float(metrics[key]) for key in METRIC_KEYS if key in metrics})
    if error is not None:
        row["error"] = str(error)[:2000]
    path = result_path(dataset, model, subset, spec["name"], pool)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(row, indent=2), encoding="utf-8")


def clean_cuda():
    gc.collect()
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


def run_one(encoder, model, dataset, subset, spec, gpu_index):
    paths = expected_paths(dataset, model, subset, spec)
    if all(is_complete(path) for path in paths):
        print(f"skip {model} {dataset}/{subset} {spec['name']}", flush=True)
        return
    splits = load_split(subset)
    cache = DATA_ROOT / "frame_cache" / f"gpu{gpu_index}" / dataset
    batch = MODEL_BATCH.get(model, BATCH)
    text_batch = MODEL_TEXT_BATCH.get(model, TEXT_BATCH)
    print(f"== {PRETTY.get(model, model)} | {dataset}/{subset} | {spec['name']} ({sampling_text(model, spec)})", flush=True)
    try:
        if spec["type"] == "single":
            scored = {
                "none": evaluate_text_to_tracklet(
                    encoder,
                    splits,
                    num_frames=spec["num_frames"],
                    batch_size=batch,
                    text_batch_size=text_batch,
                    junk_same_camera=False,
                    frame_cache=cache,
                    frame_sample=spec["frame_sample"],
                )
            }
        else:
            scored = evaluate_text_to_tracklet_windows(
                encoder,
                splits,
                num_frames=NATIVE_FRAMES.get(model, 8),
                stride=spec["stride"],
                sample_fps=spec["sample_fps"],
                max_frames=spec["max_frames"],
                pools=POOLS,
                batch_size=batch,
                text_batch_size=text_batch,
                junk_same_camera=False,
                frame_cache=cache,
            )
    except RuntimeError as exc:
        pools = ["none"] if spec["type"] == "single" else list(POOLS)
        for pool in pools:
            save_row(dataset, model, subset, spec, pool, error=exc)
        if "out of memory" in str(exc).lower():
            raise StopModel(str(exc)) from exc
        raise
    for pool, metrics in scored.items():
        save_row(dataset, model, subset, spec, pool, metrics=metrics, cached=encoder.last_hit)
        print(f"-- {pool}: R1 {metrics['Rank-1']:.2f}  mAP {metrics['mAP']:.2f}  mINP {metrics.get('mINP', float('nan')):.2f}", flush=True)

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for this benchmark")
if ON_KAGGLE and torch.cuda.device_count() < 2:
    raise RuntimeError("Select Kaggle GPU T4 x2; this notebook needs two visible GPUs")
GPU_COUNT = min(2, torch.cuda.device_count())
print(f"Using {GPU_COUNT} GPU worker(s): " + ", ".join(torch.cuda.get_device_name(i) for i in range(GPU_COUNT)), flush=True)
model_queue = Queue()
for model in MODELS:
    model_queue.put(model)


def run_gpu_worker(gpu_index):
    torch.cuda.set_device(gpu_index)
    device = f"cuda:{gpu_index}"
    ensure_cuda_healthy(device)
    while True:
        try:
            model = model_queue.get_nowait()
        except Empty:
            return
        jobs = [
            (dataset, spec, subset)
            for dataset in ("activitynet",)
            for spec in protocols_for(model)
            for subset in DATASETS["activitynet"]["subsets"]
        ]
        todo = [job for job in jobs if not all(is_complete(p) for p in expected_paths(job[0], model, job[2], job[1]))]
        print(f"\n######## GPU {gpu_index} | {PRETTY.get(model, model)}: {len(todo)} of {len(jobs)} runs to do", flush=True)
        if not todo:
            continue
        try:
            encoder = GalleryCache(build_encoder(model, device=device))
        except Exception:
            traceback.print_exc()
            print(f"Could not build {model} on GPU {gpu_index}; skipping it", flush=True)
            clean_cuda()
            continue
        try:
            for dataset, spec, subset in todo:
                run_one(encoder, model, dataset, subset, spec, gpu_index)
        except StopModel as exc:
            print(f"CUDA OOM on {model} (GPU {gpu_index}); stopping this model: {exc}", flush=True)
        finally:
            del encoder
            clean_cuda()


with ThreadPoolExecutor(max_workers=GPU_COUNT) as workers:
    futures = [workers.submit(run_gpu_worker, i) for i in range(GPU_COUNT)]
    for future in futures:
        future.result()  # propagate worker failures to the notebook

print("Done", flush=True)

Using 2 GPU worker(s): Tesla T4, Tesla T4

######## GPU 0 | SigLIP 2 So400m: 8 of 8 runs to do

######## GPU 1 | Perception Encoder L/14: 8 of 8 runs to do


preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/559 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


tokenizer_config.json:   0%|          | 0.00/47.2k [00:00<?, ?B/s]

...

Decode frames:   0%|          | 0/1000 [00:00<?, ?video/s]


xclip texts: 100%|██████████| 63/63 [00:01<00:00, 36.48batch/s]h/s]

irra videos:  93%|█████████▎| 6823/7306 [26:37<02:16,  3.54batch/s]

-- none: R1 21.20  mAP 33.98  mINP 33.98
== X-CLIP | activitynet/sentence | uniform8 (uniform 8 frames)


Decode frames:   0%|          | 0/1000 [00:00<?, ?video/s]


irra videos:  94%|█████████▍| 6851/7306 [26:41<01:00,  7.57batch/s]

In [10]:
# Run this cell directly after Step 4 setup/helpers are loaded; skip the main run cell.
# Recompute IRRA and X-CLIP even if their previous result rows say complete.
retry_models = ("irra", "xclip")
def retry_run_one(encoder, model, dataset, subset, spec, gpu_index):
    splits = load_split(subset)
    cache = DATA_ROOT / "frame_cache" / f"gpu{gpu_index}" / dataset
    batch = MODEL_BATCH.get(model, BATCH)
    text_batch = MODEL_TEXT_BATCH.get(model, TEXT_BATCH)
    print(f"== {PRETTY.get(model, model)} | {dataset}/{subset} | {spec['name']}", flush=True)
    try:
        if spec["type"] == "single":
            scored = {
                "none": evaluate_text_to_tracklet(
                    encoder, splits, num_frames=spec["num_frames"], batch_size=batch,
                    text_batch_size=text_batch, junk_same_camera=False,
                    frame_cache=cache, frame_sample=spec["frame_sample"],
                )
            }
        else:
            scored = evaluate_text_to_tracklet_windows(
                encoder, splits, num_frames=NATIVE_FRAMES.get(model, 8),
                stride=spec["stride"], sample_fps=spec["sample_fps"],
                max_frames=spec["max_frames"], pools=POOLS,
                batch_size=batch, text_batch_size=text_batch,
                junk_same_camera=False, frame_cache=cache,
            )
    except RuntimeError as exc:
        pools = ["none"] if spec["type"] == "single" else list(POOLS)
        for pool in pools:
            save_row(dataset, model, subset, spec, pool, error=exc)
        if "out of memory" in str(exc).lower():
            raise StopModel(str(exc)) from exc
        raise
    for pool, metrics in scored.items():
        save_row(dataset, model, subset, spec, pool, metrics=metrics, cached=encoder.last_hit)
        print(f"-- {pool}: R1 {metrics['Rank-1']:.2f}  mAP {metrics['mAP']:.2f}", flush=True)


def retry_gpu_worker(gpu_index, model):
    torch.cuda.set_device(gpu_index)
    device = f"cuda:{gpu_index}"
    ensure_cuda_healthy(device)
    jobs = [
        (dataset, spec, subset)
        for dataset in ("activitynet",)
        for spec in protocols_for(model)
        for subset in DATASETS["activitynet"]["subsets"]
    ]
    print(f"\n######## GPU {gpu_index} | {PRETTY.get(model, model)}: forcing all {len(jobs)} runs", flush=True)
    try:
        encoder = GalleryCache(build_encoder(model, device=device))
    except Exception:
        traceback.print_exc()
        print(f"Could not build {model} on GPU {gpu_index}; skipping it", flush=True)
        clean_cuda()
        return
    try:
        for dataset, spec, subset in jobs:
            retry_run_one(encoder, model, dataset, subset, spec, gpu_index)
    except StopModel as exc:
        print(f"CUDA OOM on {model} (GPU {gpu_index}); stopping this model: {exc}", flush=True)
    finally:
        del encoder
        clean_cuda()

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for this benchmark")
if ON_KAGGLE and torch.cuda.device_count() < 2:
    raise RuntimeError("Select Kaggle GPU T4 x2; this notebook needs two visible GPUs")
retry_gpu_count = min(2, torch.cuda.device_count())
print("Retrying IRRA on GPU 0 and X-CLIP on GPU 1 in parallel", flush=True)
with ThreadPoolExecutor(max_workers=2) as workers:
    retry_futures = [
        workers.submit(retry_gpu_worker, gpu_index, model)
        for gpu_index, model in enumerate(retry_models)
    ]
    for future in retry_futures:
        future.result()
print("IRRA and X-CLIP retry done", flush=True)

Retrying IRRA on GPU 0 and X-CLIP on GPU 1 in parallel

######## GPU 0 | IRRA: forcing all 8 runs

######## GPU 1 | X-CLIP: forcing all 8 runs


Loading weights:   0%|          | 0/591 [00:00<?, ?it/s]

Model device: cuda:1 dtype=torch.float16 requested=cuda:1 (Tesla T4)
== X-CLIP | activitynet/paragraph | uniform8


Decode frames:   0%|          | 0/1000 [00:00<?, ?video/s]

xclip videos:   6%|▌         | 14/250 [00:01<00:31,  7.51batch/s]

...

-- mean: R1 15.10  mAP 25.68
-- mean_s8: R1 15.10  mAP 25.80
-- max: R1 9.70  mAP 18.61
-- query_max: R1 16.40  mAP 27.20
== IRRA | activitynet/sentence | reid_8fps_n64


Decode windows:   0%|          | 0/1000 [00:00<?, ?video/s]

Window encode: 1000 tracklets, 14612 clips (window=8 stride=4 fps=8 max=64)


irra texts: 100%|██████████| 227/227 [00:02<00:00, 81.75batch/s]


-- mean: R1 7.33  mAP 14.13
-- mean_s8: R1 7.27  mAP 14.08
-- max: R1 4.56  mAP 10.44
-- query_max: R1 8.29  mAP 15.40
IRRA and X-CLIP retry done


## 5. Tables

Reads ActivityNet results from every results folder it can find: this session's `zero_shot_results/`, plus any attached Kaggle inputs that contain a `zero_shot_results` folder (other groups, earlier runs). Each table is shown here and saved as CSV and Markdown under `zero_shot_results/tables/`.

Accuracy cells use each model's **best Rank-1 setting** per ActivityNet subset (protocol and pool are shown in the settings table). All numbers are percentages except MdR (median rank).

In [11]:
import numpy as np

KEY = ["dataset", "subset", "model", "protocol", "pool"]


def collect_rows():
    roots = [RESULTS_ROOT]
    if Path("/kaggle/input").is_dir():
        roots += [p for p in Path("/kaggle/input").rglob("zero_shot_results") if p.is_dir()]
    rows = []
    for root in dict.fromkeys(roots):
        for path in root.rglob("*.json"):
            try:
                row = json.loads(path.read_text(encoding="utf-8"))
            except Exception:
                continue
            if not isinstance(row, dict) or "Rank-1" not in row:
                continue
            if row.get("status", "complete") != "complete" or row.get("dataset") != "activitynet":
                continue
            rows.append(row)
    if not rows:
        return pd.DataFrame(
            columns=KEY + ["Rank-1", "mAP", "mINP", "protocol_type", "gallery_cached", "Model", "Dataset",
                           "video_ms_per_item", "peak_gpu_gb", "total_s"]
        )
    frame = pd.DataFrame(rows)
    if "protocol_type" not in frame:
        frame["protocol_type"] = np.where(frame["pool"] == "none", "single", "window")
    frame["protocol_type"] = frame["protocol_type"].fillna(
        pd.Series(np.where(frame["pool"] == "none", "single", "window"), index=frame.index)
    )
    frame["created_at"] = frame.get("created_at", pd.Series(dtype=str)).fillna("")
    frame["gallery_cached"] = frame.get("gallery_cached", False)
    frame["gallery_cached"] = frame["gallery_cached"].fillna(False).astype(bool)
    frame = frame.sort_values("created_at").drop_duplicates(KEY, keep="last")
    frame["Model"] = frame["model"].map(lambda m: PRETTY.get(m, m))
    frame["Dataset"] = frame["dataset"].map(lambda d: DATASETS.get(d, {}).get("label", d))
    return frame.reset_index(drop=True)


RESULTS = collect_rows()
print(f"{len(RESULTS)} result rows from {RESULTS['model'].nunique() if len(RESULTS) else 0} models", flush=True)
BEST = (
    RESULTS.sort_values(["Rank-1", "mAP"], ascending=False, kind="mergesort")
    .groupby(["dataset", "subset", "model"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)
TABLES = {}


def column_order(dataset):
    return [s for s in DATASETS.get(dataset, {}).get("subsets", []) if s in set(BEST.loc[BEST.dataset == dataset, "subset"])]


def setting_text(row):
    return row["protocol"] if row["pool"] in ("none", None) or pd.isna(row["pool"]) else f"{row['protocol']} / {row['pool']}"

104 result rows from 4 models


### 5.1 Leaderboard per dataset (Rank-1 / mAP / mINP)

In [12]:
for dataset in [d for d in DATASETS if d in set(BEST["dataset"])]:
    part = BEST[BEST.dataset == dataset].copy()
    subsets = column_order(dataset)
    part["cell"] = part.apply(lambda r: f"{r['Rank-1']:.2f} / {r['mAP']:.2f} / {r.get('mINP', np.nan):.2f}", axis=1)
    part["setting"] = part.apply(setting_text, axis=1)
    order = part.groupby("Model")["Rank-1"].mean().sort_values(ascending=False).index
    board = part.pivot(index="Model", columns="subset", values="cell").reindex(index=order, columns=subsets)
    board["mean Rank-1"] = part.groupby("Model")["Rank-1"].mean().reindex(order).round(2)
    settings = part.pivot(index="Model", columns="subset", values="setting").reindex(index=order, columns=subsets)
    label = DATASETS[dataset]["label"]
    TABLES[f"leaderboard_{dataset}"] = board
    TABLES[f"best_setting_{dataset}"] = settings
    display(Markdown(f"#### {label}"))
    display(board)
    display(Markdown(f"Best setting per cell ({label})"))
    display(settings)

#### ActivityNet Captions

subset,paragraph,sentence,mean Rank-1
Model,,,
Perception Encoder L/14,68.10 / 78.87 / 78.87,41.91 / 54.83 / 54.83,55.01
SigLIP 2 So400m,57.10 / 70.18 / 70.18,35.11 / 48.51 / 48.51,46.11
X-CLIP,28.60 / 41.53 / 41.53,14.04 / 23.59 / 23.59,21.32
IRRA,16.40 / 27.20 / 27.20,8.29 / 15.40 / 15.40,12.35


Best setting per cell (ActivityNet Captions)

subset,paragraph,sentence
Model,,
Perception Encoder L/14,vt_2fps_n32 / query_max,reid_8fps_n64 / query_max
SigLIP 2 So400m,reid_8fps_n64 / query_max,reid_8fps_n64 / query_max
X-CLIP,vt_2fps_n32 / mean,reid_8fps_n64 / mean
IRRA,reid_8fps_n64 / query_max,reid_8fps_n64 / query_max


### 5.2 Protocol and pool comparison (top three models, Rank-1)

In [13]:
for dataset in [d for d in DATASETS if DATASETS[d]["protocols"] == "video" and d in set(RESULTS["dataset"])]:
    part = RESULTS[RESULTS.dataset == dataset].copy()
    top = part.groupby("Model")["Rank-1"].max().sort_values(ascending=False).head(3).index
    part = part[part.Model.isin(top)]
    part["pool"] = part["pool"].replace({"none": "-"})
    table = part.pivot_table(index=["Model", "protocol", "pool"], columns="subset", values="Rank-1", aggfunc="max")
    table = table.reindex(columns=column_order(dataset)).round(2)
    table = table.reindex(top, level=0)
    TABLES[f"protocols_{dataset}"] = table
    display(Markdown(f"#### {DATASETS[dataset]['label']}"))
    display(table)

#### ActivityNet Captions

subset                                           paragraph  sentence
Model                   protocol      pool                          
Perception Encoder L/14 reid_8fps_n64 max             58.5     33.62
                                      mean            64.8     38.73
                                      mean_s8         65.3     38.93
                                      query_max       67.5     41.91
                        uniform8      -               62.1     36.25
                        vt_1fps_n12   max             63.1     37.43
                                      mean            63.2     37.30
                                      mean_s8         63.2     37.30
                                      query_max       64.9     37.79
                        vt_2fps_n32   max             64.0     36.80
                                      mean            64.8     38.62
                                      mean_s8         64.9     38.40
                                      query_max       68.1     41.17
SigLIP 2 So400m         reid_8fps_n64 max             50.4     28.12
                                      mean            54.4     31.08
                                      mean_s8         53.9     30.69
                                      query_max       57.1     35.11
                        uniform8      -               51.8     28.50
                        vt_1fps_n12   max             51.8     29.97
                                      mean            52.2     29.91
                                      mean_s8         52.2     29.91
                                      query_max       53.2     30.61
                        vt_2fps_n32   max             52.9     29.89
                                      mean            54.8     30.94
                                      mean_s8         54.4     30.58
                                      query_max       56.1     33.78
X-CLIP                  reid_8fps_n64 max             15.3      7.41
                                      mean            27.7     14.04
                                      mean_s8         27.9     13.91
                                      query_max       25.1     12.75
                        uniform8      -               21.2      9.81
                        vt_1fps_n12   max             23.8     10.98
                                      mean            23.7     11.89
                                      mean_s8         23.7     11.89
                                      query_max       23.1     11.50
                        vt_2fps_n32   max             18.6      9.21
                                      mean            28.6     13.38
                                      mean_s8         28.4     13.22
                                      query_max       25.8     13.19

### 5.3 ActivityNet Captions: paragraph vs sentence queries

Same gallery, different queries. Paragraph queries describe the whole video and are mostly longer than the 32 to 77 token text window of the CLIP-style encoders (the tail is truncated); sentence queries describe one event of the video.

In [14]:
anet = BEST[BEST.dataset == "activitynet"]
if len(anet):
    gap = anet.pivot(index="Model", columns="subset", values="Rank-1").reindex(columns=["paragraph", "sentence"])
    gap["paragraph - sentence"] = gap["paragraph"] - gap["sentence"]
    gap = gap.sort_values("paragraph", ascending=False).round(2)
    TABLES["activitynet_paragraph_vs_sentence"] = gap
    display(gap)
else:
    print("No ActivityNet rows yet")

subset,paragraph,sentence,paragraph - sentence
Model,,,
Perception Encoder L/14,68.1,41.91,26.19
SigLIP 2 So400m,57.1,35.11,21.99
X-CLIP,28.6,14.04,14.56
IRRA,16.4,8.29,8.11


### 5.4 Speed and GPU (Tesla T4)

In [15]:
if len(RESULTS):
    fresh = RESULTS[~RESULTS["gallery_cached"]]
    single = fresh[fresh["protocol_type"] == "single"]
    windows = fresh[fresh["protocol_type"] == "window"]
    speed = pd.DataFrame(
        {
            "ms / clip (single clip)": single.groupby("Model")["video_ms_per_item"].median(),
            "ms / clip (windows)": windows.groupby("Model")["video_ms_per_item"].median(),
            "peak GPU GB": RESULTS.groupby("Model")["peak_gpu_gb"].max(),
            "total minutes": RESULTS.groupby("Model")["total_s"].sum() / 60.0,
            "rows": RESULTS.groupby("Model").size(),
        }
    )
    speed = speed.sort_values("ms / clip (single clip)").round(2)
    TABLES["speed_gpu"] = speed
    display(speed)

,ms / clip (single clip),ms / clip (windows),peak GPU GB,total minutes,rows
Model,,,,,
X-CLIP,30.76,29.94,0.45,49.25,26
IRRA,75.02,76.75,0.61,90.68,26
Perception Encoder L/14,374.51,375.85,1.32,639.64,26
SigLIP 2 So400m,486.44,486.05,2.23,822.98,26


### 5.5 Save tables

In [16]:
def markdown_table(frame):
    frame = frame.reset_index() if not isinstance(frame.index, pd.RangeIndex) else frame
    headers = [str(c) for c in frame.columns]

    def cell(value):
        if isinstance(value, float):
            return "-" if np.isnan(value) else f"{value:.2f}"
        return str(value)

    lines = ["| " + " | ".join(headers) + " |", "|" + "|".join("---" for _ in headers) + "|"]
    for _, row in frame.iterrows():
        lines.append("| " + " | ".join(cell(v) for v in row) + " |")
    return "\n".join(lines)


TABLE_DIR = RESULTS_ROOT / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS.to_csv(TABLE_DIR / "all_rows.csv", index=False)
DATASET_SUMMARY.to_csv(TABLE_DIR / "datasets.csv", index=False)
report = ["# Zero-shot results", "", "## Datasets", "", markdown_table(DATASET_SUMMARY.drop(columns=["root"]))]
for name, table in TABLES.items():
    table.to_csv(TABLE_DIR / f"{name}.csv")
    report += ["", f"## {name}", "", markdown_table(table)]
(TABLE_DIR / "report.md").write_text("\n".join(report) + "\n", encoding="utf-8")
print("Wrote", sorted(p.name for p in TABLE_DIR.iterdir()), flush=True)

Wrote ['activitynet_paragraph_vs_sentence.csv', 'all_rows.csv', 'best_setting_activitynet.csv', 'datasets.csv', 'leaderboard_activitynet.csv', 'protocols_activitynet.csv', 'report.md', 'speed_gpu.csv']
